In [1]:
!pip install -q sentence-transformers chromadb pypdf langchain langchain-community faiss-cpu transformers torch numpy pandas

import subprocess
result = subprocess.run(['pip', 'list'], capture_output=True, text=True)
print("Installed packages:")
for line in result.stdout.split('\n'):
    if any(pkg in line for pkg in ['sentence-transformers', 'chromadb', 'langchain', 'faiss']):
        print(f"  {line.strip()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231

In [4]:
import os
import json
import time
import numpy as np
from typing import List, Dict, Any
from datetime import datetime

from pypdf import PdfReader

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Updated import for modern langchain versions
from langchain_core.documents import Document

from tqdm.notebook import tqdm

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
sample_research_content = """
Title: Deep Learning for Natural Language Processing: A Comprehensive Survey

Abstract: This paper presents a comprehensive survey of deep learning techniques applied to natural language processing tasks. We examine the evolution from traditional statistical methods to modern neural architectures, focusing on transformer-based models and their applications.

1. Introduction
Natural Language Processing (NLP) has undergone a revolutionary transformation with the advent of deep learning. Traditional rule-based systems have given way to neural networks that learn patterns from vast amounts of text data.

2. Background: Traditional NLP Methods
Before deep learning, NLP relied heavily on:
- Bag-of-words models
- N-gram language models
- Hidden Markov Models (HMMs)
- Conditional Random Fields (CRFs)

3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for long dependencies
- Convolutional Neural Networks (CNNs) for local patterns

4. Transformer Architecture
The transformer model, introduced in "Attention is All You Need," revolutionized NLP through:
- Self-attention mechanisms
- Parallel processing capabilities
- Scalability to large datasets

5. Large Language Models
Modern LLMs like BERT, GPT, and T5 demonstrate:
- Few-shot learning capabilities
- Transfer learning effectiveness
- Emergent behaviors at scale

6. Applications and Future Directions
Current applications include machine translation, question answering, and text generation. Future research focuses on efficiency, interpretability, and reducing computational requirements.
"""

with open('sample_research_paper.txt', 'w') as f:
    f.write(sample_research_content)

print("Sample research paper created!")
print("File: sample_research_paper.txt")
print(f"Size: {len(sample_research_content)} characters")

Sample research paper created!
File: sample_research_paper.txt
Size: 1696 characters


In [6]:
class PDFProcessor:
    """Handles PDF reading and text processing"""

    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", ". ", "! ", "? "]
        )

    def load_pdf_content(self, file_path: str) -> str:
        """Load content from PDF or text file"""
        try:
            if file_path.endswith('.pdf'):
                reader = PdfReader(file_path)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() + "\n"
            else:
                with open(file_path, 'r') as f:
                    text = f.read()
            print(f"Loaded {len(text)} characters from {file_path}")
            return text
        except Exception as e:
            print(f"Error loading file {file_path}: {e}")
            return ""

    def create_documents(self, text: str, source_name: str) -> List[Document]:
        """Convert text into LangChain documents"""
        chunks = self.text_splitter.split_text(text)

        documents = []
        for i, chunk in enumerate(chunks):
            doc = Document(
                page_content=chunk,
                metadata={"source": source_name, "chunk": i + 1}
            )
            documents.append(doc)
        print(f"Created {len(documents)} documents from {source_name}")
        return documents

processor = PDFProcessor()

documents = processor.create_documents(sample_research_content, "research_paper_survey")

print("\nFirst 3 chunks:")
for i, doc in enumerate(documents[:3]):
    print(f"\n --- Chunk {i+1} --- ")
    print(f"Content: {doc.page_content[:200]} ... ")
    print(f"Size: {len(doc.page_content)} characters")

Created 4 documents from research_paper_survey

First 3 chunks:

 --- Chunk 1 --- 
Content: Title: Deep Learning for Natural Language Processing: A Comprehensive Survey

Abstract: This paper presents a comprehensive survey of deep learning techniques applied to natural language processing ta ... 
Size: 359 characters

 --- Chunk 2 --- 
Content: 1. Introduction
Natural Language Processing (NLP) has undergone a revolutionary transformation with the advent of deep learning. Traditional rule-based systems have given way to neural networks that l ... 
Size: 442 characters

 --- Chunk 3 --- 
Content: 3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for lon ... 
Size: 490 characters


In [7]:
class EmbeddingManager:
    """Handles text embeddings using open-source models"""

    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = 384

        print(f"Loaded embedding model: all-MiniLM-L6-v2")
        print(f"Embedding dimension: {self.embedding_dim}")

    def create_embeddings(self, texts: List[str]) -> np.ndarray:
        """Convert texts to embedding vectors"""
        print("Creating embeddings ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Created {len(embeddings)} embeddings")
        return embeddings

embed_manager = EmbeddingManager()

test_texts = ["Machine learning is amazing", "Deep learning uses neural networks", "AI transforms the world"]
test_query = "neural networks"
embeddings = embed_manager.create_embeddings(test_texts)
print(f"Created {len(embeddings)} embeddings for test texts")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: all-MiniLM-L6-v2
Embedding dimension: 384
Creating embeddings ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created 3 embeddings
Created 3 embeddings for test texts


In [8]:
class VectorStoreManager:
    """Manages the in-memory vector database"""
    def __init__(self, embedding_manager: EmbeddingManager):
        self.embedding_manager = embedding_manager

        self.client = chromadb.Client()

        self.collection = self.client.create_collection(
            name="research_papers",
            metadata={"description": "Academic paper chunks"}
        )
        print("Created in-memory vector store")

    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store"""
        print("Adding documents to vector store ... ")

        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]

        embeddings = self.embedding_manager.create_embeddings(texts)

        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        print(f"Added {len(documents)} documents to vector store")

    def search(self, query: str, n_results: int = 3) -> List[Dict[str, Any]]:
        """Search for similar documents"""
        query_embedding = self.embedding_manager.create_embeddings([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        formatted_results = []
        for i in range(len(results['documents'][0])):
            formatted_results.append({
                'content': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })
        return formatted_results

vector_store = VectorStoreManager(embed_manager)
vector_store.add_documents(documents)

search_results = vector_store.search("What is deep learning?", n_results=2)
print("\nSearch Results:")
for i, result in enumerate(search_results, 1):
    print(f"\n{i}. Content: {result['content'][:200]} ... ")
    print(f"Distance: {result['distance']:.3f}")

Created in-memory vector store
Adding documents to vector store ... 
Creating embeddings ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created 4 embeddings
Added 4 documents to vector store
Creating embeddings ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created 1 embeddings

Search Results:

1. Content: Title: Deep Learning for Natural Language Processing: A Comprehensive Survey

Abstract: This paper presents a comprehensive survey of deep learning techniques applied to natural language processing ta ... 
Distance: 1.147

2. Content: 3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for lon ... 
Distance: 1.151


In [12]:
from transformers import pipeline

class OpenSourceLLM:
    """Open-source language model for Q&A"""

    def __init__(self):
        model_name = "google/flan-t5-small"

        print("Loading open-source language model ... ")
        self.qa_pipeline = pipeline(
            "text-generation",
            model=model_name,
            device=-1
        )
        print(f"Loaded model: {model_name}")

    def generate_answer(self, question: str, context: str) -> str:
        """Generate answer using context"""
        prompt = f"Answer the question based on the context provided.\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"

        response = self.qa_pipeline(prompt, max_new_tokens=200, do_sample=True)
        return response[0]['generated_text']

llm = OpenSourceLLM()

test_context = "Deep learning is a subset of machine learning that uses neural networks with multiple layers."
test_question = "What is deep learning?"
test_answer = llm.generate_answer(test_question, test_context)
print(f"\nTest Answer: {test_answer}")

Loading open-source language model ... 


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Loaded model: google/flan-t5-small

Test Answer: Answer the question based on the context provided.

Context: Deep learning is a subset of machine learning that uses neural networks with multiple layers.

Question: What is deep learning?
Answer:wa1 >= 30Su4[d], R|W] [6|4w2:rn* += 30] # 3A1 [18.9 =: 2()]-vk2 0ejb_-Cu20,4, 1


In [13]:
from transformers import pipeline

class ResearchAssistant:
    """Complete RAG system for research papers"""

    def __init__(self, vector_store: VectorStoreManager, llm: OpenSourceLLM):
        self.vector_store = vector_store
        self.llm = llm
        self.conversation_history = []

    def ask_question(self, question: str, n_contexts: int = 3) -> Dict[str, Any]:
        """Ask a question about the research paper"""
        print(f"Processing: {question}")

        start_time = time.time()

        relevant_docs = self.vector_store.search(question, n_results=n_contexts)

        combined_context = "\n\n".join([doc['content'] for doc in relevant_docs])

        answer = self.llm.generate_answer(question, combined_context)

        processing_time = time.time() - start_time

        result = {
            "question": question,
            "answer": answer,
            "contexts_used": len(relevant_docs),
            "processing_time": round(processing_time, 2),
            "sources": [doc['metadata'] for doc in relevant_docs]
        }

        self.conversation_history.append(result)

        return result

assistant = ResearchAssistant(vector_store, llm)

print("Research Assistant is ready!")
print("You can now ask questions about the research paper.")

Research Assistant is ready!
You can now ask questions about the research paper.


In [14]:
educational_questions = [
    "What is the difference between traditional NLP and deep learning NLP?",
    "Can you explain what a transformer is in simple terms?",
    "What are the main applications of deep learning in NLP?",
    "How do neural networks help with language understanding?",
    "What comes before deep learning in NLP history?"
]

print("🎓 Educational Questions & Answers:")
print("=" * 50)

for question in educational_questions:
    print(f"\nQuestion: {question}")
    result = assistant.ask_question(question)
    print(f"Answer: {result['answer']}")
    print(f"Processing time: {result['processing_time']}s")
    print(f"Sources used: {result['contexts_used']} chunks")

🎓 Educational Questions & Answers:

Question: What is the difference between traditional NLP and deep learning NLP?
Processing: What is the difference between traditional NLP and deep learning NLP?
Creating embeddings ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created 1 embeddings


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Answer the question based on the context provided.

Context: 1. Introduction
Natural Language Processing (NLP) has undergone a revolutionary transformation with the advent of deep learning. Traditional rule-based systems have given way to neural networks that learn patterns from vast amounts of text data.

2. Background: Traditional NLP Methods
Before deep learning, NLP relied heavily on:
- Bag-of-words models
- N-gram language models
- Hidden Markov Models (HMMs)
- Conditional Random Fields (CRFs)

3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for long dependencies
- Convolutional Neural Networks (CNNs) for local patterns

4. Transformer Architecture
The transformer model, introduced in "Attention is All You Need," revolutionized NLP through:
- Self-attention mechanisms
- Parallel processing capabilities
- Scalability to large datas

...

Answer: Answer the question based on the context provided.

Context: 3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for long dependencies
- Convolutional Neural Networks (CNNs) for local patterns

4. Transformer Architecture
The transformer model, introduced in "Attention is All You Need," revolutionized NLP through:
- Self-attention mechanisms
- Parallel processing capabilities
- Scalability to large datasets

1. Introduction
Natural Language Processing (NLP) has undergone a revolutionary transformation with the advent of deep learning. Traditional rule-based systems have given way to neural networks that learn patterns from vast amounts of text data.

2. Background: Traditional NLP Methods
Before deep learning, NLP relied heavily on:
- Bag-of-words models
- N-gram language models
- Hidden Markov Models (HMMs)
- Conditional Random Fields (CR

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Created 1 embeddings


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Answer the question based on the context provided.

Context: 1. Introduction
Natural Language Processing (NLP) has undergone a revolutionary transformation with the advent of deep learning. Traditional rule-based systems have given way to neural networks that learn patterns from vast amounts of text data.

2. Background: Traditional NLP Methods
Before deep learning, NLP relied heavily on:
- Bag-of-words models
- N-gram language models
- Hidden Markov Models (HMMs)
- Conditional Random Fields (CRFs)

3. Neural Network Foundations
Deep learning in NLP builds upon several key neural architectures:
- Recurrent Neural Networks (RNNs) for sequential data
- Long Short-Term Memory (LSTM) networks for long dependencies
- Convolutional Neural Networks (CNNs) for local patterns

4. Transformer Architecture
The transformer model, introduced in "Attention is All You Need," revolutionized NLP through:
- Self-attention mechanisms
- Parallel processing capabilities
- Scalability to large datas

In [15]:
print("🎉 Congratulations! You've built a complete GenAI system!")
print("\nWhat you learned:")
print("How to process PDF documents into searchable chunks")
print("Using open-source embedding models (no API keys!)")
print("Building in-memory vector databases with ChromaDB")
print("Creating Q&A systems with open-source language models")
print("Adding educational features for better learning")

print("\n Next steps to explore:")
print("1. Try with your own PDF research papers")
print("2. Experiment with different embedding models")
print("3. Add conversation memory for follow-up questions")
print("4. Create a web interface using Streamlit")
print("5. Try larger open-source models like Llama-2")

with open('learning_session.json', 'w') as f:
    json.dump(assistant.conversation_history, f, indent=2)

print("\nConversation history saved to 'learning_session.json'")

🎉 Congratulations! You've built a complete GenAI system!

What you learned:
How to process PDF documents into searchable chunks
Using open-source embedding models (no API keys!)
Building in-memory vector databases with ChromaDB
Creating Q&A systems with open-source language models
Adding educational features for better learning

 Next steps to explore:
1. Try with your own PDF research papers
2. Experiment with different embedding models
3. Add conversation memory for follow-up questions
4. Create a web interface using Streamlit
5. Try larger open-source models like Llama-2

Conversation history saved to 'learning_session.json'
